# Test bert-base wrong TrGLUE examples on remaining models

Runs inference **only on the wrong examples** collected from bert-base-turkish-cased on TrGLUE MNLI (matched and mismatched). Uses the **same models and loading as** `src/base_results`:
- **Qwen2-7B-Instruct** (generation; Qwen chat template via `apply_chat_template`)
- **Gemma-3-27b-it** (generation; Gemma chat template via `apply_chat_template`)
- **mDeBERTa-v3-base-mnli-xnli** (sequence classification, premise [SEP] hypothesis — no generation)

Matched and mismatched are kept separate. Results saved per model per split as `wrong_predictions_{model_short}_{split}.json`.

In [ ]:
# Installs (uncomment if running on Colab / fresh env)
# !pip install -q transformers datasets accelerate scikit-learn tqdm torch

import json
import re
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Constants
BATCH_SIZE = 8
MAX_NEW_TOKENS = 5
DO_SAMPLE = False
TEMPERATURE = 0.0

# Directory containing wrong_examples_TrGLUE-MNLI_test_matched.json and wrong_examples_TrGLUE-MNLI_test_mismatched.json
_cwd = Path.cwd()
if (_cwd / "bert-base-turkish-cased").exists():
    RESULTS_DIR = _cwd / "bert-base-turkish-cased"
else:
    RESULTS_DIR = _cwd / "src" / "model_distillation" / "bert-base-turkish-cased"
assert RESULTS_DIR.exists(), f"RESULTS_DIR not found: {RESULTS_DIR}"

# Aligned with base_results: Qwen2-7B-Instruct, Gemma-3-27b-it only, mDeBERTa
MODELS = [
    {"name": "Qwen/Qwen2-7B-Instruct", "short": "Qwen2-7B-Instruct", "type": "generation"},
    {"name": "google/gemma-3-27b-it", "short": "gemma-3-27b-it", "type": "generation"},
    {"name": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", "short": "mDeBERTa-v3-base-mnli", "type": "classifier"},
]

WRONG_MATCHED_PATH = RESULTS_DIR / "wrong_examples_TrGLUE-MNLI_test_matched.json"
WRONG_MISMATCHED_PATH = RESULTS_DIR / "wrong_examples_TrGLUE-MNLI_test_mismatched.json"

print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"Matched path exists: {WRONG_MATCHED_PATH.exists()}")
print(f"Mismatched path exists: {WRONG_MISMATCHED_PATH.exists()}")

In [ ]:
# Load wrong examples: keep matched and mismatched separate
with open(WRONG_MATCHED_PATH, "r", encoding="utf-8") as f:
    wrong_matched = json.load(f)
with open(WRONG_MISMATCHED_PATH, "r", encoding="utf-8") as f:
    wrong_mismatched = json.load(f)

print(f"Wrong matched:   {len(wrong_matched)} examples")
print(f"Wrong mismatched: {len(wrong_mismatched)} examples")

# Normalize label string -> int for ground truth (JSON has true_label as string)
LABEL_STR_TO_ID = {"entailment": 0, "neutral": 1, "contradiction": 2}

def true_label_id(ex):
    s = (ex.get("true_label") or "neutral").strip().lower()
    return LABEL_STR_TO_ID.get(s, 1)

# Build lists of (premise, hypothesis, true_label_id) per split
def to_inputs(items):
    return [
        (x["premise"], x["hypothesis"], true_label_id(x))
        for x in items
    ]

inputs_matched = to_inputs(wrong_matched)
inputs_mismatched = to_inputs(wrong_mismatched)

In [ ]:
LABEL_NAMES = ["entailment", "neutral", "contradiction"]

def parse_nli_label(raw_text: str, prompt_prefix: str = "") -> int:
    """Robust parsing: first word after output, lowercase, strip punctuation.
    Maps English and Turkish. Default to neutral (1) on failure.
    """
    if prompt_prefix:
        text = raw_text[len(prompt_prefix):].strip() if raw_text.startswith(prompt_prefix) else raw_text.strip()
    else:
        text = raw_text.strip()
    text = text.lower().strip()
    if not text:
        return 1
    # First token: strip punctuation
    first = text.split()[0] if text.split() else ""
    first = re.sub(r"^[^a-zçğıöşüA-ZÇĞİÖŞÜ0-9]+", "", first)
    first = re.sub(r"[^a-zçğıöşüA-ZÇĞİÖŞÜ0-9]+$", "", first)
    first = first.lower()
    # English
    label_map = {
        "entailment": 0,
        "neutral": 1,
        "contradiction": 2,
        "içerme": 0,
        "nötr": 1,
        "tarafsız": 1,
        "çelişki": 2,
    }
    return label_map.get(first, 1)


def predict_on_wrong(model_info, inputs_list, split_name, out_dir: Path):
    """Run model on wrong examples. model_info has 'name', 'short', 'type' (generation | classifier)."""
    model_name = model_info["name"]
    short = model_info["short"]
    is_classifier = model_info["type"] == "classifier"

    if is_classifier:
        # mDeBERTa: same as base_results — classifier, no chat template
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, ignore_mismatched_sizes=True)
        model = model.to(device)
        model.eval()
    else:
        # Generation: same as base_results — tokenizer.apply_chat_template with system+user
        # (Qwen2 uses Qwen chat template; Gemma-3 uses Gemma template; each tokenizer applies its own)
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if getattr(tokenizer, "pad_token", None) is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32, trust_remote_code=True)
        model = model.to(device)
        model.eval()

    premises = [x[0] for x in inputs_list]
    hypotheses = [x[1] for x in inputs_list]
    y_true = np.array([x[2] for x in inputs_list])

    if is_classifier:
        # NLI classification: premise [SEP] hypothesis
        y_pred = []
        for start in tqdm(range(0, len(inputs_list), BATCH_SIZE), desc=f"{short} {split_name}"):
            batch_p = premises[start : start + BATCH_SIZE]
            batch_h = hypotheses[start : start + BATCH_SIZE]
            enc = tokenizer(batch_p, batch_h, truncation=True, max_length=256, padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            with torch.no_grad():
                logits = model(**enc).logits
            preds = logits.argmax(-1).cpu().numpy()
            y_pred.extend(preds.tolist())
        y_pred = np.array(y_pred)
        raw_texts = [LABEL_NAMES[p] for p in y_pred]
    else:
        # Generation: use tokenizer.apply_chat_template (Qwen format for Qwen2, Gemma format for Gemma-3)
        def nli_messages(p, h):
            return [
                {"role": "system", "content": "You are a helpful assistant for natural language inference. Classify the relationship between premise and hypothesis as entailment, neutral, or contradiction. Respond with exactly one word: entailment, neutral, or contradiction. No explanation."},
                {"role": "user", "content": f"Premise: {p}\nHypothesis: {h}\nLabel:"},
            ]

        y_pred = []
        raw_texts = []
        for start in tqdm(range(0, len(inputs_list), BATCH_SIZE), desc=f"{short} {split_name}"):
            end = min(start + BATCH_SIZE, len(inputs_list))
            batch_inputs = inputs_list[start:end]
            batch_prompts = [nli_messages(x[0], x[1]) for x in batch_inputs]
            formatted = tokenizer.apply_chat_template(
                batch_prompts,
                tokenize=False,
                add_generation_prompt=True,
            )
            if isinstance(formatted, str):
                formatted = [formatted]
            enc = tokenizer(formatted, return_tensors="pt", padding=True, truncation=True, max_length=512)
            enc = {k: v.to(device) for k, v in enc.items()}
            with torch.no_grad():
                out = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE if DO_SAMPLE else None,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
            # Decode only new tokens (skip input length) for parsing
            input_lens = enc["attention_mask"].sum(dim=1)
            for i in range(out.size(0)):
                gen_only = out[i, input_lens[i]:]
                gen_text = tokenizer.decode(gen_only, skip_special_tokens=True)
                raw_texts.append(gen_text)
                pred_id = parse_nli_label(gen_text, "")
                y_pred.append(pred_id)
        y_pred = np.array(y_pred)

    acc = float(accuracy_score(y_true, y_pred))

    # Debug: first 5 examples
    print(f"\n[{short}] {split_name} — first 5 raw + parsed:")
    for i in range(min(5, len(raw_texts))):
        r = raw_texts[i]
        r_show = (r[:120] + "...") if len(r) > 120 else r
        print(f"  {i+1}. raw: {r_show} -> parsed: {LABEL_NAMES[y_pred[i]]}")

    # Save per-example predictions
    out_dir.mkdir(parents=True, exist_ok=True)
    save_path = out_dir / f"wrong_predictions_{short}_{split_name}.json"
    records = []
    for idx, (prem, hyp, true_id) in enumerate(inputs_list):
        records.append({
            "premise": prem,
            "hypothesis": hyp,
            "true_label": LABEL_NAMES[true_id],
            "predicted_label": LABEL_NAMES[int(y_pred[idx])],
            "predicted_id": int(y_pred[idx]),
        })
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump({"accuracy": acc, "n": len(y_true), "predictions": records}, f, ensure_ascii=False, indent=2)
    print(f"Saved: {save_path}")
    return acc, y_true, y_pred


def _unload_model():
    """Free GPU memory before loading next model."""
    import gc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# Output directory for wrong_predictions_*.json
OUT_DIR = Path(RESULTS_DIR)  # same as wrong examples, or set to e.g. Path("results_wrong")
summary = []  # (model_short, split, accuracy)

In [ ]:
# Main loop: each model on matched and mismatched wrongs (separate)
for model_info in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_info['name']}")
    print("="*60)

    acc_matched, _, _ = predict_on_wrong(model_info, inputs_matched, "matched", OUT_DIR)
    summary.append((model_info["short"], "matched", acc_matched))

    acc_mismatched, _, _ = predict_on_wrong(model_info, inputs_mismatched, "mismatched", OUT_DIR)
    summary.append((model_info["short"], "mismatched", acc_mismatched))

    # Free memory before next model
    _unload_model()

In [ ]:
# Summary: accuracy per model per split on these hard examples
print("\n" + "="*60)
print("Summary — Accuracy on bert-base wrong TrGLUE examples")
print("="*60)
for short, split, acc in summary:
    print(f"  {short}  {split:12}  {acc:.4f}")
print("="*60)